# Praktikum Pertemuan 3 — Menyiapkan Data Proyek agar Layak dan Aman Dianalisis

**Mata Kuliah:** Exploratory Data Analysis (KMTI21133) — Pertemuan 3  
**Fase PjBL:** Perencanaan  
**Sub-CPMK3:** Mahasiswa mampu merumuskan pertanyaan analisis dari masalah nyata serta memperoleh data
sesuai lisensi sumbernya, serta mengidentifikasi dan menangani data pribadi yang terkandung di dalamnya (C4, A3).

**Cara kerja.** Sel bertanda `# TODO` Anda kerjakan sendiri; setiap TODO disertai *expected output* agar Anda bisa memeriksa jawaban sendiri. Sel Markdown bertanda **Jawaban Anda** diisi dengan tulisan, bukan kode.

**Bagian A–F** dikerjakan di kelas dengan dataset latihan. **Bagian G** dikerjakan bersama tim di luar kelas dengan dataset proyek Anda sendiri, dan hasilnya menjadi isi **M1 (bobot 5%)** yang dikumpulkan sesuai tenggat di LMS.

## Bagian 0 — Persiapan

Dataset latihan: **`peserta_pelatihan_umkm_2025.csv`** — 620 baris, 28 kolom.
Data pendaftaran dan hasil pelatihan UMKM binaan kampus.

> **Catatan penting:** seluruh isi dataset ini **sintetis** dan dibuat untuk latihan. Tidak ada orang nyata di dalamnya.
> Nomor NIK, nomor telepon, dan surel sengaja dibuat dengan pola yang tidak valid. Perlakukan tetap seolah-olah data nyata,
> karena itulah kebiasaan yang sedang kita latih.

In [2]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
print('pandas', pd.__version__)

pandas 2.2.3


In [ ]:
FILE = 'peserta_pelatihan_umkm_2025.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()          # pilih peserta_pelatihan_umkm_2025.csv
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)
print(df.shape)
df.head(3)

---
## Bagian A — Merumuskan pertanyaan analisis

Sebuah pertanyaan analisis dianggap layak bila lolos **tiga uji**:

1. **Uji kolom** — kolom mana yang akan menjawabnya?
2. **Uji bentuk jawaban** — jawabannya berupa angka, peringkat, perbandingan, atau sebaran?
3. **Uji tindak lanjut** — kalau terjawab, apa yang bisa diputuskan?

In [ ]:
# TODO A1 — kenali kolom yang tersedia sebelum menulis pertanyaan
import pandas as pd

# 1. Memuat dataset
df = pd.read_csv('peserta_pelatihan_umkm_2025.csv')

# a. Cetak daftar nama kolom beserta tipe datanya
print("--- DAFTAR KOLOM DAN TIPE DATA ---")
for kolom, tipe_data in df.dtypes.items():
    print(f"- {kolom}: {tipe_data}")

# b. Cetak jumlah baris dan kolom
print("\n--- UKURAN DATASET ---")
jumlah_baris, jumlah_kolom = df.shape
print(f"Expected: {jumlah_baris} baris x {jumlah_kolom} kolom")

### TODO A2 — perbaiki tiga pertanyaan berikut

Tulis versi perbaikannya di sel di bawah, beserta kolom yang akan dipakai.

| No | Draf pertanyaan | Masalahnya |
|---|---|---|
| 1 | Bagaimana kondisi peserta pelatihan? | Terlalu kabur |
| 2 | Apakah peserta yang lebih kaya lebih sukses? | "Kaya" dan "sukses" belum didefinisikan sebagai kolom |
| 3 | Apakah pelatihan menaikkan omzet peserta? | Uji kolom gagal — cek sendiri mengapa |

**Jawaban Anda:**

1. Perbaikan: Bagaimana distribusi profil demografi peserta pelatihan berdasarkan tingkat pendidikan terakhir dan penghasilan pribadi bulanan? — Kolom: pendidikan_terakhir, penghasilan_pribadi_bulanan

2. Perbaikan: Apakah terdapat hubungan antara tingkat penghasilan pribadi bulanan dengan omzet usaha bulanan peserta? — Kolom: penghasilan_pribadi_bulanan, omzet_usaha_bulanan

3. Mengapa uji kolom gagal: Karena di dalam dataset hanya tersedia satu kolom omzet (omzet_usaha_bulanan) yang diambil saat pendaftaran (sebelum pelatihan), dan tidak ada kolom data omzet sesudah pelatihan, sehingga analisis perbandingan atau uji kenaikan omzet sebelum dan sesudah pelatihan tidak dapat dilakukan.

In [ ]:
# TODO A3 — buktikan jawaban nomor 3 dengan kode
import pandas as pd

# 1. Memuat dataset
df = pd.read_csv('peserta_pelatihan_umkm_2025.csv')

# 2. Cari kolom yang mengandung kata 'omzet'
kolom_omzet = [col for col in df.columns if 'omzet' in col.lower()]

print("Kolom omzet yang ditemukan:", kolom_omzet)

# Expected: hanya ada satu kolom omzet, yaitu omzet saat mendaftar

---
## Bagian B — Skala pengukuran

### TODO B1 — klasifikasikan delapan kolom berikut

Isi tabel di sel Markdown ini. Pilihan: **nominal · ordinal · interval · rasio**.

| Kolom | Skala | Alasan singkat |
|---|---|---|
| `kategori_usaha` | | |
| `pendidikan_terakhir` | | |
| `skor_prates` | | |
| `omzet_usaha_bulanan` | | |
| `jumlah_karyawan` | | |
| `tingkat_kepuasan` | | |
| `tanggal_daftar` | | |
| `kecamatan` | | |

In [ ]:
# TODO B2 — perbaiki satu kolom ordinal yang salah tipe
import pandas as pd

# a. Tentukan urutan kategori pendidikan
kategori_pendidikan = ['SD', 'SMP', 'SMA/SMK', 'D3', 'S1', 'S2']

# Ubah kolom 'pendidikan_terakhir' menjadi tipe kategori berurutan (ordered categorical)
df['pendidikan_terakhir'] = pd.Categorical(
    df['pendidikan_terakhir'],
    categories=kategori_pendidikan,
    ordered=True
)

# b. Tampilkan value_counts() yang sudah urut menurut jenjang
print("--- VALUE COUNTS ---")
print(df['pendidikan_terakhir'].value_counts())

# c. Cetak median jenjang pendidikan lewat .cat.codes
median_code = df['pendidikan_terakhir'].cat.codes.median()
median_jenjang = kategori_pendidikan[int(median_code)]

print("\n--- MEDIAN JENJANG PENDIDIKAN ---")
print(f"Median code: {median_code}")
print(f"Median = {median_jenjang}")

In [ ]:
# TODO B3 — ubah dua kolom tanggal menjadi bertipe datetime
# Kolom: tanggal_lahir dan tanggal_daftar
# Lalu cetak tanggal pendaftaran paling awal dan paling akhir
# Expected: rentang pendaftaran berada di tahun 2025


---
## Bagian C — Kamus data

Kamus data adalah satu baris penjelasan untuk setiap kolom. Bangkitkan kerangkanya dengan kode, isi maknanya sendiri.

In [ ]:
# TODO C1 — bangkitkan kerangka kamus data
import pandas as pd

# Memuat dataset
df = pd.read_csv('peserta_pelatihan_umkm_2025.csv')

# Menyiapkan list untuk menampung informasi setiap kolom
data_kamus = []

for col in df.columns:
    tipe_data = str(df[col].dtype)
    nilai_unik = df[col].nunique()
    nilai_kosong = df[col].isnull().sum()
    contoh_nilai = str(df[col].dropna().iloc[0]) if len(df[col].dropna()) > 0 else ""

    data_kamus.append({
        'nama_kolom': col,
        'tipe_data': tipe_data,
        'nilai_unik': nilai_unik,
        'nilai_kosong': nilai_kosong,
        'contoh_nilai': contoh_nilai,
        'deskripsi': '',
        'skala_pengukuran': '',
        'kategori_data_pribadi': '',
        'tindakan_penanganan': ''
    })

# Membuat DataFrame 'kamus'
kamus = pd.DataFrame(data_kamus)

# Menampilkan informasi ukuran dan beberapa baris pertama
print(f"Expected: {kamus.shape[0]} baris x {kamus.shape[1]} kolom")
display(kamus.head())

In [ ]:
# TODO C2 — kolom mana yang punya nilai kosong?
# Tampilkan hanya baris kamus yang nilai_kosong-nya lebih dari 0
kolom_kosong = kamus[kamus['nilai_kosong'] > 0]

# Menampilkan kolom tertentu agar mudah dibaca
display(kolom_kosong[['nama_kolom', 'tipe_data', 'nilai_kosong', 'contoh_nilai']])

### TODO C3 — isi kamus untuk lima kolom

Lengkapi tabel berikut. Deskripsi ditulis sebagai kalimat lengkap, bukan pengulangan nama kolom.

| Kolom | Deskripsi | Skala | Satuan |
|---|---|---|---|
| `lama_usaha_tahun` | Menunjukkan lamanya waktu usaha telah berjalan dan beroperasi sejak pertama kali didirikan hingga saat pendaftaran pelatihan. | Rasio | Tahun |
| `kehadiran_persen` | Menyatakan tingkat kehadiran peserta selama sesi pelatihan berlangsung dalam bentuk persentase dari total keseluruhan pertemuan. | Rasio | Persen (%) |
| `skor_pascates` | Menggambarkan tingkat pemahaman dan penguasaan materi peserta yang diukur melalui tes akhir setelah pelatihan selesai dilaksanakan. | Rasio | Poin (1-100) |
| `kanal_pendaftaran` | Menunjukkan sarana atau media yang digunakan oleh peserta untuk mendaftarkan diri secara resmi ke program pelatihan UMKM. | Nominal | Kategori (Teks) |
| `jumlah_anak` | Menyatakan banyaknya anak yang menjadi tanggungan dalam keluarga peserta pelatihan. | Rasio | Orang (Jiwa) |

---
## Bagian D — Menandai data pribadi

| Kategori | Artinya |
|---|---|
| **pengenal langsung** | Sendirian sudah menunjuk ke satu orang |
| **quasi-identifier** | Sendirian aman, gabungannya menunjuk ke satu orang |
| **data spesifik** | Bocornya merugikan lebih berat (kesehatan, keuangan pribadi, data anak) |
| **bukan data pribadi** | Tidak menunjuk ke orang; inilah bahan analisis |

In [ ]:
# TODO D1 — susun tabel penandaan Anda sendiri
import pandas as pd

# 1. Definisikan dictionary PENANDAAN
# Format: 'nama_kolom': ('kategori', 'tindakan')
PENANDAAN = {
    'id_peserta': ('pengenal langsung', 'hash'),
    'nama_lengkap': ('pengenal langsung', 'hapus'),
    'nik_dummy': ('pengenal langsung', 'hash'),
    'no_hp': ('pengenal langsung', 'hash'),
    'email': ('pengenal langsung', 'hash'),
    'alamat_jalan': ('data spesifik', 'generalisasi'),
    'tanggal_lahir': ('data spesifik', 'generalisasi'),
    'tanggal_daftar': ('data spesifik', 'pertahankan')
}

# 2. Bangun DataFrame 'tanda'
data_tanda = []
for col in df.columns:
    if col in PENANDAAN:
        kat, tind = PENANDAAN[col]
    else:
        kat, tind = 'bukan data pribadi', 'pertahankan'

    data_tanda.append({
        'nama_kolom': col,
        'kategori': kat,
        'tindakan': tind
    })

tanda = pd.DataFrame(data_tanda)

# 3. Cetak jumlah kolom per kategori
print("--- JUMLAH KOLOM PER KATEGORI ---")
print(tanda['kategori'].value_counts())

# Menampilkan tabel lengkapnya
display(tanda)

### TODO D2 — pertanggungjawabkan tiga keputusan

Tiga kolom berikut sering diperdebatkan. Tulis keputusan Anda **beserta alasannya**.

| Kolom | Kategori pilihan Anda | Tindakan | Alasan |
|---|---|---|---|
| `nama_usaha` | Bukan data pribadi | Pertahankan | Nama usaha (terutama UMKM) bersifat publik dan bertujuan untuk komersial agar dapat ditemukan pelanggan melalui mesin pencari, sehingga tidak melanggar privasi individu pemiliknya. |
| `omzet_usaha_bulanan` | Bukan data pribadi | Pertahankan | Omzet bulanan merupakan data operasional atau finansial dari badan usaha, bukan informasi pribadi privat perorangan, sehingga penting dipertahankan untuk analisis bisnis. |
| `id_peserta` | Bukan data pribadi | Hash | Merupakan kode unik yang merujuk langsung pada individu peserta; perlu di-hash untuk menjaga anonimitas identitas personal saat analisis, namun tetap memungkinkan identifikasi baris data secara unik tanpa membocorkan identitas asli. |

> Petunjuk untuk `nama_usaha`: bayangkan seseorang mencari nama itu di mesin pencari.
> Petunjuk untuk `omzet_usaha_bulanan`: apakah ini data tentang **orang** atau tentang **usaha**?
> Petunjuk untuk `id_peserta`: apakah Anda masih perlu membedakan satu baris dari baris lain?

---
## Bagian E — Menerapkan tiga tindakan

In [ ]:
# TODO E1 — HAPUS
# Buat DataFrame baru bernama 'aman' yang sudah membuang semua kolom bertindakan 'hapus'.
# Cetak berapa kolom yang dibuang dan berapa yang tersisa.
# 1. Mendapatkan daftar kolom yang memiliki tindakan 'hapus'
kolom_hapus = tanda[tanda['tindakan'] == 'hapus']['nama_kolom'].tolist()

# 2. Membuat DataFrame baru bernama 'aman' tanpa kolom yang dihapus
aman = df.drop(columns=kolom_hapus)

# 3. Mencetak jumlah kolom yang dibuang dan yang tersisa
print(f"Jumlah kolom yang dibuang: {len(kolom_hapus)}")
print(f"Jumlah kolom yang tersisa di DataFrame 'aman': {aman.shape[1]}")


In [ ]:
# TODO E2 — HASH
# Tulis fungsi samarkan(nilai, garam) yang mengembalikan 12 karakter pertama
# dari sha256(garam + nilai). Kembalikan np.nan bila nilainya kosong.
# Terapkan pada kolom yang Anda beri tindakan 'hash'.
# Expected: nilai asli dan hasil hash berbeda, dan nilai asli yang sama menghasilkan hash yang sama

import pandas as pd
import numpy as np
import hashlib

PENANDAAN = {
    'id_peserta': ('pengenal langsung', 'hash'),
    'nama_lengkap': ('pengenal langsung', 'hapus'),
    'nik_dummy': ('pengenal langsung', 'hash'),
    'no_hp': ('pengenal langsung', 'hash'),
    'email': ('pengenal langsung', 'hash'),
    'alamat_jalan': ('data spesifik', 'generalisasi'),
    'tanggal_lahir': ('data spesifik', 'generalisasi'),
    'tanggal_daftar': ('data spesifik', 'pertahankan')
}

data_tanda = []
for col in df.columns:
    if col in PENANDAAN:
        kat, tind = PENANDAAN[col]
    else:
        kat, tind = 'bukan data pribadi', 'pertahankan'
    data_tanda.append({'nama_kolom': col, 'kategori': kat, 'tindakan': tind})
tanda = pd.DataFrame(data_tanda)

kolom_hapus = tanda[tanda['tindakan'] == 'hapus']['nama_kolom'].tolist()
aman = df.drop(columns=kolom_hapus)

GARAM = 'ganti-dengan-kalimat-rahasia-tim'

def samarkan(nilai, garam):
    if pd.isna(nilai):
        return np.nan
    # Pastikan nilai dikonversi ke string
    teks = str(nilai).strip()
    gabung = str(garam) + teks
    h = hashlib.sha256(gabung.encode('utf-8')).hexdigest()
    return h[:12]

kolom_hash = tanda[tanda['tindakan'] == 'hash']['nama_kolom'].tolist()
print("Kolom yang akan di-hash:", kolom_hash)

for col in kolom_hash:
    if col in aman.columns:
        aman[col] = aman[col].apply(lambda x: samarkan(x, GARAM))

# Cek hasil
print(aman[kolom_hash].head())


In [ ]:
# TODO E3 — buktikan hash bersifat konsisten dan tidak terbaca
# a. Hash dua kali nilai yang sama, tunjukkan hasilnya identik
nilai_contoh = "ratna.hidayat@contoh.example"
hasil_1 = samarkan(nilai_contoh, GARAM)
hasil_2 = samarkan(nilai_contoh, GARAM)

print(f"Nilai Asli: {nilai_contoh}")
print(f"Hash ke-1: {hasil_1}")
print(f"Hash ke-2: {hasil_2}")
print(f"Apakah identik? {hasil_1 == hasil_2}\n")

# b. Hash dua nilai berbeda, tunjukkan hasilnya berbeda
nilai_A = "ratna.hidayat@contoh.example"
nilai_B = "bunga.santoso@contoh.example"
hash_A = samarkan(nilai_A, GARAM)
hash_B = samarkan(nilai_B, GARAM)

print(f"Nilai A: {nilai_A} -> Hash: {hash_A}")
print(f"Nilai B: {nilai_B} -> Hash: {hash_B}")
print(f"Apakah berbeda? {hash_A != hash_B}")


**Jawaban Anda:** kalau `GARAM` ikut ter-*commit* ke GitHub, apa yang masih terlindungi dan apa yang tidak?

Apa yang Masih Terlindungi?

Data Asli (Plaintext): Nilai asli dari data pribadi yang sensitif (seperti nomor telepon, email, NIK, atau ID peserta) tetap aman dan tidak terbaca secara langsung di dalam repositori karena telah diubah menjadi string hash satu arah (sha256).

Apa yang TIDAK Terlindungi?

Keamanan Hash terhadap Serangan Brute-Force / Rainbow Table: Karena algoritma hashing bersifat deterministik (nilai yang sama selalu menghasilkan hash yang sama), siapa pun yang mengetahui string GARAM dapat melakukan teknik pencocokan tebakan (brute-force atau dictionary attack) dengan membandingkan daftar kemungkinan data asli ditambah GARAM hingga menemukan hasil hash yang cocok. Oleh karena itu, kerahasiaan salt (garam) sangat krusial untuk mencegah pembalikan enkripsi hash secara offline.

In [ ]:
import pandas as pd
import numpy as np

# A. Hitung usia peserta pada tanggal acuan 1 September 2025
ACUAN = pd.Timestamp('2025-09-01')
aman['tanggal_lahir'] = pd.to_datetime(aman['tanggal_lahir'])
# Menghitung selisih hari lalu dibagi 365.25 untuk mengakomodasi tahun kabisat
aman['usia'] = (ACUAN - aman['tanggal_lahir']).dt.days // 365.25

# B. Buat kolom 'kelompok_umur' dengan batas: <25, 25-34, 35-44, 45-54, 55+
bins = [-np.inf, 24.99, 34.99, 44.99, 54.99, np.inf]
labels = ['<25', '25-34', '35-44', '45-54', '55+']
aman['kelompok_umur'] = pd.cut(aman['usia'], bins=bins, labels=labels)

print("--- DISTRIBUSI KELOMPOK UMUR ---")
print(aman['kelompok_umur'].value_counts())

# C. Buang kolom tanggal_lahir dan kelurahan dari 'aman'
kolom_buang = ['tanggal_lahir', 'kelurahan']
aman = aman.drop(columns=[col for col in kolom_buang if col in aman.columns])

# D. Rapikan kolom kecamatan yang sebagian tertulis KAPITAL
# Menggunakan .str.title() agar format kapitalisasi seragam (huruf kapital di awal kata)
aman['kecamatan'] = aman['kecamatan'].str.strip().str.title()

print("\n--- KECAMATAN SETELAH DIRAPIKAN ---")
print(f"Jumlah nilai unik kecamatan: {aman['kecamatan'].nunique()}")
print(aman['kecamatan'].unique())

---
## Bagian F — Memeriksa sisa risiko

Menghapus nama belum tentu cukup. Untuk setiap kombinasi quasi-identifier, hitung berapa orang yang punya kombinasi sama. Nilai terkecil disebut **k**. Target kelas ini: **k ≥ 5**.

In [ ]:
import pandas as pd

def cek_k(data, kolom):
    """
    Fungsi untuk mengevaluasi k-anonymity pada kolom atau kombinasi kolom tertentu.
    Mengembalikan Series berisi: k (grup terkecil), grup berisi 1 orang, dan total grup.
    """
    # Menghitung ukuran setiap grup (mendukung kolom tunggal maupun list/kombinasi kolom)
    if isinstance(kolom, list):
        grup_counts = data.groupby(kolom).size()
    else:
        grup_counts = data[kolom].value_counts()

    # k adalah ukuran grup terkecil (frekuensi minimum)
    k = grup_counts.min() if len(grup_counts) > 0 else 0

    # Jumlah grup yang hanya berisi 1 orang (frekuensi == 1)
    grup_1_orang = (grup_counts == 1).sum()

    # Total keseluruhan grup unik
    total_grup = len(grup_counts)

    return pd.Series({
        'k': k,
        'grup_1_orang': grup_1_orang,
        'total_grup': total_grup
    })

# Contoh penggunaan fungsi pada DataFrame 'aman' untuk kolom 'kecamatan'
print(cek_k(aman, 'kecamatan'))

In [ ]:
# TODO F2 — ukur risiko SEBELUM penanganan
# Gunakan data asli df dengan kolom:
#   ['kelurahan', 'jenis_kelamin', 'pendidikan_terakhir', 'tanggal_lahir']
# Expected: k = 1 dan SELURUH 620 baris berada di grup berisi satu orang
# Tentukan kombinasi kolom kuasi-identifikasi yang diuji
kolom_risiko = ['kelurahan', 'jenis_kelamin', 'pendidikan_terakhir', 'tanggal_lahir']

# Panggil fungsi cek_k pada dataframe asli (df)
hasil_sebelum = cek_k(df, kolom_risiko)

print("--- RISIKO K-ANONYMITY SEBELUM PENANGANAN ---")
print(hasil_sebelum)

**Jawaban Anda:** keempat kolom itu tidak memuat nama maupun nomor telepon. Jelaskan dalam dua kalimat mengapa hasilnya tetap berbahaya.

Kombinasi atribut seperti tanggal lahir, kelurahan, jenis kelamin, dan pendidikan terakhir berfungsi sebagai kuasi-identifikator unik yang dapat mempersempit pencarian hingga ke satu orang spesifik tanpa memerlukan nama atau nomor telepon. Hal ini memungkinkan penyerang menghubungkan data tersebut dengan sumber informasi publik atau eksternal untuk membongkar identitas asli individu dengan mudah.

In [ ]:
# TODO F3 — ukur risiko SESUDAH generalisasi
# Gunakan 'aman' dengan kolom:
#   ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'kelompok_umur']
# Expected: jumlah grup berisi satu orang turun banyak, tetapi k masih 1
# Tentukan kolom kuasi-identifikasi yang sudah digeneralisisasi pada DataFrame 'aman'
kolom_risiko_sesudah = ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'kelompok_umur']

# Uji fungsi cek_k dengan parameter observed=True (jika menggunakan kolom kategori)
def cek_k_aman(data, kolom):
    grup_counts = data.groupby(kolom, observed=True).size()
    k = grup_counts.min() if len(grup_counts) > 0 else 0
    grup_1_orang = (grup_counts == 1).sum()
    total_grup = len(grup_counts)
    return pd.Series({'k': k, 'grup_1_orang': grup_1_orang, 'total_grup': total_grup})

print("--- RISIKO K-ANONYMITY SESUDAH GENERALISASI ---")
print(cek_k_aman(aman, kolom_risiko_sesudah))

In [ ]:
import pandas as pd
import numpy as np

# Load and prepare aman
aman = df.copy()

ACUAN = pd.Timestamp('2025-09-01')
aman['tanggal_lahir'] = pd.to_datetime(aman['tanggal_lahir'])
aman['usia'] = (ACUAN - aman['tanggal_lahir']).dt.days // 365.25

# Current bins: [-inf, 24.99, 34.99, 44.99, 54.99, inf] -> '<25', '25-34', '35-44', '45-54', '55+'
bins = [-np.inf, 24.99, 34.99, 44.99, 54.99, np.inf]
labels = ['<25', '25-34', '35-44', '45-54', '55+']
aman['kelompok_umur'] = pd.cut(aman['usia'], bins=bins, labels=labels)
aman['kecamatan'] = aman['kecamatan'].str.strip().str.title()

def cek_k_aman(data, kolom):
    grup_counts = data.groupby(kolom, observed=True).size()
    k = grup_counts.min() if len(grup_counts) > 0 else 0
    grup_1_orang = (grup_counts == 1).sum()
    total_grup = len(grup_counts)
    return pd.Series({'k': k, 'grup_1_orang': grup_1_orang, 'total_grup': total_grup})

cols = ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'kelompok_umur']
print("Initial state with current bins:")
print(cek_k_aman(aman, cols))

# Let's see what groups have size 1
grup_counts = aman.groupby(cols, observed=True).size()
satu_orang = grup_counts[grup_counts == 1]
print("\nSome groups with size 1:")
print(satu_orang.head(10))

**Jawaban Anda:** tulis kombinasi yang Anda pilih, nilai k yang dicapai, dan informasi apa yang Anda korbankan untuk mencapainya.

Kombinasi yang dipilih adalah Memperlebar rentang kelompok umur (menjadi <30, 30-49, dan 50+) serta menggabungkan kecamatan dengan populasi kecil ke dalam kategori 'Lainnya'. Melalui strategi ini, nilai $k \ge 5$ berhasil dicapai secara keseluruhan pada kombinasi kuasi-identifikator.Informasi yang dikorbankan untuk mencapai tingkat privasi tersebut adalah tingkat kejelasan detail demografis yang lebih spesifik (granularitas usia menjadi lebih lebar dan hilangnya identitas nama kecamatan pinggiran secara terpisah), namun hal ini dilakukan demi menjaga keamanan data peserta agar tidak ada kombinasi unik yang dapat diidentifikasi.

In [ ]:
import pandas as pd

# a. Buat tabel silang jumlah peserta menurut kecamatan x kategori_usaha
tabel_silang = pd.crosstab(aman['kecamatan'], aman['kategori_usaha'])

print("--- TABEL SILANG KECAMATAN x KATEGORI USAHA ---")
display(tabel_silang)

# b. Hitung berapa sel yang terisi tetapi berisi kurang dari 5 peserta (antara 1 sampai 4)
sel_kecil = ((tabel_silang > 0) & (tabel_silang < 5)).sum().sum()

print(f"\nJumlah sel yang terisi namun bernilai kecil (1-4 peserta): {sel_kecil}")

---
## Bagian G — Diterapkan pada proyek tim (dikerjakan di luar kelas)

Bagian ini adalah isi **M1**. Kerjakan bersama tim pada dataset proyek Anda sendiri.

### G1 — Pertanyaan analisis
Tulis **3–5 pertanyaan** yang lolos tiga uji. Untuk setiap pertanyaan, sebutkan kolom yang akan dipakai.

pertanyaan besar:
Apakah rata-rata pendapatan pariwisata (Revenue) dipengaruhi oleh pengunjung dan rating di berbagai negara?

pertanyaan:
1. Bagaimana rata-rata jumlah pengunjung (Visitors) berbeda antara destinasi yang memiliki ketersediaan akomodasi (Accommodation_Available: Yes) dan yang tidak (No)?
2. Faktor apa yang paling berkaitan dengan penilaian wisatawan (Rating), berdasarkan kategori wisata (Category) dan pendapatan (Revenue)?
3. Berapa banyak destinasi wisata yang memiliki rating di atas 4.0, dan bagaimana sebarannya menurut negara (Country)?

### G2 — Provenans data
Catat lima butir berikut di README repositori tim:

| Butir | Isi |
|---|---|
| Nama sumber | |
| Tautan atau nama instansi | |
| Tanggal pengambilan | |
| Lisensi atau nomor surat izin | |
| Kalimat sitasi untuk laporan | |

### G3 — Kamus data
Isi `TEMPLATE_Kamus_Data.csv` untuk **setiap** kolom dataset tim, termasuk kolom `kategori_data_pribadi` dan `tindakan_penanganan`. Simpan sebagai `data_dictionary.md` atau `.csv` di repositori.

In [1]:
# G4 — jalankan penanganan pada data tim dan periksa nilai k
# Salin pola Bagian E dan F, ganti dataset dan nama kolomnya.
# Simpan hasilnya ke data/processed/ dan JANGAN unggah data/raw/

### G5 — Daftar periksa sebelum `git push`

- [ ] `data/raw/` masuk `.gitignore`
- [ ] Tidak ada nama, NIK, nomor telepon, surel, atau alamat di berkas mana pun
- [ ] Keluaran sel yang sempat menampilkan data pribadi sudah dibersihkan
- [ ] Garam hash tidak tertulis di dalam notebook
- [ ] Kamus data memuat kategori dan tindakan untuk setiap kolom
- [ ] Nilai k pada data yang diunggah sudah dihitung dan dicatat
- [ ] Bila masih ragu: repositori dibuat **privat**

> Menghapus berkas lalu *commit* ulang **tidak** menghilangkan data dari riwayat Git. > Periksa **sebelum** *commit* pertama.